# 🏦 Enterprise Banking Data Science — End-to-End Experiments (Colab & Kaggle Ready)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akmalkhaniub/enterprise-banking-ds-platform/blob/main/notebooks/01_End_to_End_Banking_Experiments.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/akmalkhaniub/enterprise-banking-ds-platform/blob/main/notebooks/01_End_to_End_Banking_Experiments.ipynb)

**Author:** Akmal Khan, PhD  
**Domain Focus:** Commercial & Retail Banking Data Science — *Time-Series Forecasting, Customer 360 Segmentation, Peer-Adjusted Branch Benchmarking, AML Anomaly Detection, Analytical SQL & Controlled Enterprise GenAI*

---
### 🧪 Experiment Roadmap in This Notebook
1. **Environment Setup & Repository Clone** (Works out-of-the-box on Google Colab, Kaggle Kernels, or Local Jupyter)
2. **Experiment 1: Analytical SQL Feature Store & Automated Data Quality Gates (`DuckDB`)**
3. **Experiment 2: Statistical Hypothesis Testing (`scipy.stats` & `statsmodels` ADF / Welch's *t*-test / ANOVA)**
4. **Experiment 3: Treasury & ATM Cash Time-Series Forecasting (`SARIMAX` vs `GradientBoosting` vs `Hybrid Ensemble`)**
5. **Experiment 4: Peer-Adjusted Branch Efficiency Benchmarking (`statsmodels` OLS Regression Residuals)**
6. **Experiment 5: Customer 360 RFM Clustering (`K-Means` + `Gaussian Mixture Models`) & Isolation Forest AML Anomaly Detection**
7. **Experiment 6: Controlled Enterprise GenAI Executive Narrator (PII Guardrails & Model Limitations)**

In [1]:
# Step 1: Auto-clone repository if running inside Google Colab or Kaggle
import os, sys
from pathlib import Path

if not Path('src/pipeline_runner.py').exists() and not Path('../src/pipeline_runner.py').exists():
    !git clone https://github.com/akmalkhaniub/enterprise-banking-ds-platform.git
    os.chdir('enterprise-banking-ds-platform')
    !pip install -q duckdb statsmodels scikit-learn pandas scipy pydantic
elif Path('../src/pipeline_runner.py').exists():
    os.chdir('..')

sys.path.insert(0, str(Path.cwd()))
print('Environment ready! Running inside enterprise-banking-ds-platform.')

Environment ready! Running inside enterprise-banking-ds-platform.


## 📊 Experiment 1: Analytical SQL Feature Engineering & Data Quality Gates (`DuckDB`)
We generate **30 commercial bank branches**, **1,200 CASA & SME customer accounts**, **180 days of daily branch liquidity**, and **45,000+ RAAST / IBFT / ATM / POS transactions**, then validate them with 4 automated SQL Data Quality assertions.

In [2]:
import pandas as pd
from src.data_generator import generate_banking_dataset
from src.sql_pipeline import BankingSQLPipeline

datasets = generate_banking_dataset(seed=42)
sql_engine = BankingSQLPipeline()
sql_engine.ingest_dataframes(datasets)

dq_checks = sql_engine.run_data_quality_suite()
print('=== SQL DATA QUALITY GATES ===')
for c in dq_checks:
    print(f"{c['check_id']:<29} | Table: {c['table_name']:<27} | Failed: {c['failed_records']} / {c['total_records']:<5} | Status: {c['status']}")

df_customer_360 = sql_engine.build_customer_360_rfm_features()
df_txn_velocity = sql_engine.build_transaction_velocity_features()
df_branch_summary = sql_engine.build_branch_performance_summary()
print(f'\nCustomer 360 Feature Store Shape: {df_customer_360.shape}')
print(f'Transaction Velocity Window Store Shape: {df_txn_velocity.shape}')

=== SQL DATA QUALITY GATES ===
DQ-01-CUSTOMER-INTEGRITY      | Table: dim_customer                | Failed: 0 / 1200  | Status: PASS
DQ-02-TRANSACTION-VALIDITY    | Table: fact_transactions           | Failed: 0 / 45000 | Status: PASS
DQ-03-LIQUIDITY-INVARIANT     | Table: fact_branch_daily_liquidity | Failed: 0 / 5400  | Status: PASS
DQ-04-TIMESERIES-CONTINUITY   | Table: fact_branch_daily_liquidity | Failed: 0 / 30    | Status: PASS

Customer 360 Feature Store Shape: (1200, 22)
Transaction Velocity Window Store Shape: (45000, 22)


## 🔬 Experiment 2: Translating Business Questions into Statistical Hypothesis Tests
We test 3 core banking hypotheses using `scipy.stats` and `statsmodels`:
1. **Stationarity (ADF Test):** Does daily ATM cash demand have a unit root?
2. **Campaign A/B Uplift (Welch's Two-Sample *t*-test):** Did branches enrolled in the Digital Self-Onboarding Kiosk campaign generate significantly higher 180-day deposit inflows?
3. **Salary Window Surge (Welch's *t*-test):** Do the 1st–6th and 28th–31st salary days drive a statistically significant spike in ATM cash replenishment demand?

In [3]:
from src.forecasting_and_benchmarking import run_stationarity_and_hypothesis_tests

hyp_results = run_stationarity_and_hypothesis_tests(datasets['fact_branch_daily_liquidity'], df_branch_summary)
adf = hyp_results['adf_stationarity_test']
print(f"ADF Statistic: {adf['adf_statistic']} (p = {adf['p_value']})")
for h in hyp_results['hypothesis_tests']:
    print(f"[{h['hypothesis_id']}] Uplift/Stat: {h['effect_size_uplift_pct']}% | p-value: {h['p_value']} -> {h['decision']}")

ADF Statistic: -4.0041 (p = 0.001389)
[HYP-01-DIGITAL-CAMPAIGN] Uplift: +14.21% | p-value: 0.059195 -> FAIL_TO_REJECT_NULL
[HYP-02-SALARY-WEEK-ATM-SURGE] Uplift: +41.71% | p-value: 0.0 -> REJECT_NULL_SIGNIFICANT
[HYP-03-BRANCH-TIER-ANOVA] F-stat: 0.7053 | p-value: 0.557531 -> FAIL_TO_REJECT_NULL


## 📈 Experiment 3: Time-Series Forecasting (`SARIMAX` + `GradientBoosting` Ensemble) & Branch OLS Benchmarking
We perform a 30-day walk-forward out-of-time validation comparing a 7-Day Seasonal Naive baseline, `statsmodels` **SARIMAX(1,0,1)x(1,0,1,7)** with exogenous calendar/KIBOR variables, `scikit-learn` **GradientBoostingRegressor** with lag features, and our **Hybrid Ensemble**.

In [4]:
from src.forecasting_and_benchmarking import build_time_series_forecasts, benchmark_branches_regression

df_forecasts, fc_metrics = build_time_series_forecasts(datasets['fact_branch_daily_liquidity'], forecast_horizon_days=30)
print('=== 30-DAY WALK-FORWARD FORECASTING BENCHMARK ===')
for model_name, m in fc_metrics['models'].items():
    print(f"{model_name:<29} | MAPE: {m['mape_pct']}% | R2: {m['r2_score']}")

df_branch_bench, ols_summary = benchmark_branches_regression(df_branch_summary)
print(f"\nPeer-Adjusted Branch Benchmarking OLS R²: {ols_summary['r_squared']}")
df_branch_bench[['efficiency_rank', 'branch_id', 'branch_name', 'city', 'efficiency_index_pct', 'benchmark_status']].head(8)

=== 30-DAY WALK-FORWARD FORECASTING BENCHMARK ===
Seasonal_Naive_7D_Baseline    | MAPE: 20.44% | R2: -0.5775
Statsmodels_SARIMAX_Exog      | MAPE: 3.99% | R2: 0.9524
Sklearn_GradientBoosting_Lag  | MAPE: 2.58% | R2: 0.9792
Hybrid_SARIMAX_GBR_Ensemble   | MAPE: 3.02% | R2: 0.9732

Peer-Adjusted Branch Benchmarking OLS R²: 0.85


## 🎯 Experiment 4: Customer 360 RFM Clustering (`K-Means` + `GMM`) & AML Anomaly Detection (`Isolation Forest`)
We segment the 1,200 customers into 5 banking personas with Next-Best-Action recommendations and scan all 45,000 transactions for AML structuring (< 2.5M PKR reporting threshold) and midnight account takeover velocity drains.

In [5]:
from src.segmentation_and_anomaly import segment_customers_rfm_clustering, detect_transaction_anomalies

df_seg, seg_summary = segment_customers_rfm_clustering(df_customer_360, n_clusters=5)
df_scored_txns, anom_metrics = detect_transaction_anomalies(df_txn_velocity, contamination=0.018)

print(f"Clustering Silhouette Score (K=5): {seg_summary['silhouette_score']}")
print(f"Anomaly Detection F1 Score: {anom_metrics['benchmark_evaluation_vs_injected_fraud']['f1_score']} | ROC-AUC: {anom_metrics['benchmark_evaluation_vs_injected_fraud']['roc_auc']}")
print(f"Flagged High-Risk Transactions: {anom_metrics['total_anomalies_flagged']} (Exposed Volume: PKR {anom_metrics['total_exposed_anomaly_volume_pkr']:,.2f})")
pd.DataFrame(seg_summary['segment_profiles'])[['customer_segment', 'customer_count', 'avg_recency_days', 'avg_casa_balance_pkr', 'avg_annual_clv_pkr', 'recommended_action']]

Clustering Silhouette Score (K=5): 0.3026
Anomaly Detection F1 Score: 0.9139 | ROC-AUC: 0.9996
Flagged High-Risk Transactions: 679 (Exposed Volume: PKR 3,617,599,095.21)


## 🛡️ Experiment 5: Controlled Enterprise GenAI Narrator & PII Redaction Guardrail
We verify that National IDs (`42101-1234567-1`), IBANs (`PK36ABCD...`), and raw customer IDs are automatically masked before prompt synthesis, and generate the structured ALCO & Retail Executive Brief.

In [6]:
from src.genai_narrator import sanitize_pii_text
from src.pipeline_runner import run_full_banking_ds_pipeline

sample_raw = 'Customer CUST-200145 with CNIC 42101-1234567-1 and IBAN PK36ABCD0000001123456789'
print('Sanitized Sample:', sanitize_pii_text(sample_raw))

bundle = run_full_banking_ds_pipeline(persist_to_disk=False)
brief = bundle['executive_brief']
print('Executive Brief Governance Mode:', brief['governance_mode'])
print(f"Recommendations Generated: {len(brief['recommendations'])} | Model Assumption Blocks: {len(brief['model_assumptions_and_limitations'])}")

Sanitized Sample: Customer CUST-200*** with CNIC [REDACTED-CNIC] and IBAN [REDACTED-IBAN]
Executive Brief Governance Mode: DETERMINISTIC_SCHEMA_GROUNDED_ZERO_HALLUCINATION
Recommendations Generated: 4 | Model Assumption Blocks: 4
